# Failure Modes & Reward Hacking Analysis

This notebook produces:
1. **Failure Mode Analysis** — cases where training makes generation *worse*
2. **Reward Hacking Detection** — high reward but visually bad (CLIP vs VLM disagreement)
3. **Epoch-by-Epoch Progression** — how generation quality evolves across checkpoints
4. **Reward Signal Case Study** — reward curves vs actual visual quality

---

## Step 0: Configuration

In [ ]:
import os

# ============================================================
# >>> EDIT THESE BEFORE RUNNING <<<
# ============================================================

# SiliconFlow API Key
SILICONFLOW_API_KEY = "sk-xxx"  # <-- Replace with your key
os.environ["SILICONFLOW_API_KEY"] = SILICONFLOW_API_KEY

# VLM model for evaluation
VLM_MODEL = "Qwen/Qwen2.5-VL-32B-Instruct"

# Baseline model
BASELINE_MODEL = "deepseek-ai/Janus-Pro-1B"

# LoRA weights - final checkpoint
LORA_WEIGHTS_PATH = "assets/trained_lora"
# Change to your trained path if different:
# LORA_WEIGHTS_PATH = "/content/drive/MyDrive/T2I-RL/outputs/grpo_full/final_checkpoint"

# Epoch checkpoints (for progression analysis)
# These should be directories containing adapter_config.json + adapter_model.safetensors
EPOCH_CHECKPOINTS = {
    "Epoch 0": "outputs/grpo_siliconflow_quick/checkpoint-epoch-0",
    "Epoch 1": "outputs/grpo_siliconflow_quick/checkpoint-epoch-1",
    "Epoch 2": "outputs/grpo_siliconflow_quick/checkpoint-epoch-2",
    "Final": "outputs/grpo_siliconflow_quick/final_checkpoint",
}
# If checkpoints are not available, set to empty dict:
# EPOCH_CHECKPOINTS = {}

# Training history JSON
TRAINING_HISTORY_PATH = "assets/training_history.json"
# Or: TRAINING_HISTORY_PATH = "outputs/grpo_siliconflow_quick/training_history.json"

# Output directory
OUTPUT_DIR = "failure_mode_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

SEED = 42
USE_4BIT = True

print("Configuration loaded.")
print(f"  Epoch checkpoints: {len(EPOCH_CHECKPOINTS)}")

## Step 1: Install Dependencies & Clone Repo

In [ ]:
!nvidia-smi
!pip install -q torch==2.4.0 torchvision==0.19.0 --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers==4.49.0 accelerate safetensors
!pip install -q --no-deps git+https://github.com/deepseek-ai/Janus.git
!pip install -q attrdict sentencepiece open-clip-torch peft bitsandbytes
!pip install -q tqdm Pillow matplotlib seaborn pandas openai
print("\n✓ Dependencies installed")

In [ ]:
import os
if not os.path.exists('T2I-RL-Project'):
    !git clone https://github.com/Inoriany/T2I-RL-Project.git
os.chdir('T2I-RL-Project')

import sys
sys.path.insert(0, 'src')

print(f"Working directory: {os.getcwd()}")

## Step 2: Load Models

In [ ]:
import torch
import numpy as np
from models.generators import JanusProGenerator, GenerationConfig
from models.reward_models import CLIPRewardModel
from openai import OpenAI
from PIL import Image
from tqdm.auto import tqdm
import json, re, base64, time
from io import BytesIO
import matplotlib.pyplot as plt
import pandas as pd

# Load baseline
print("Loading baseline Janus-Pro-1B...")
baseline_gen = JanusProGenerator(
    model_name_or_path=BASELINE_MODEL,
    device="cuda",
    load_in_4bit=USE_4BIT,
)
baseline_gen.load_model()
baseline_gen.eval()
print("✓ Baseline model loaded\n")

# Load trained
print("Loading trained model...")
trained_gen = JanusProGenerator(
    model_name_or_path=BASELINE_MODEL,
    device="cuda",
    load_in_4bit=USE_4BIT,
)
trained_gen.load_model()
trained_gen.enable_lora(lora_path=LORA_WEIGHTS_PATH)
trained_gen.eval()
print("✓ Trained model loaded\n")

# Load CLIP
clip_reward = CLIPRewardModel(device="cuda")
clip_reward.load_model()
print("✓ CLIP reward model loaded\n")

# VLM client
vlm_client = OpenAI(
    api_key=os.environ["SILICONFLOW_API_KEY"],
    base_url="https://api.siliconflow.cn/v1"
)
print("✓ VLM client ready")

---
## Part 1: Failure Mode Detection

Find prompts where the trained model generates **worse** images than baseline.

We test with a diverse set of prompts and use both CLIP + VLM to evaluate.

In [ ]:
# Diverse prompts designed to stress-test the model
# Including edge cases that RL training might struggle with

FAILURE_TEST_PROMPTS = [
    # --- Standard compositional prompts (should improve) ---
    "a red apple and a blue cup",
    "three cats on a sofa",
    "a bird above the tree",
    "a large elephant and a small mouse",
    
    # --- Complex scenes (might fail) ---
    "a chef cooking in a kitchen with pots on a stove and vegetables on the counter",
    "three children building a sandcastle on a beach with blue ocean waves",
    "a busy intersection in a modern city at night with neon signs",
    
    # --- Unusual / creative prompts (potential reward hacking) ---
    "a transparent glass vase with colorful flowers",
    "a reflection of mountains in a still lake",
    "a shadow of a person on a white wall",
    
    # --- Negation / rare concepts (hard for T2I) ---
    "an empty room with white walls",
    "a plain white plate on a wooden table",
    "a single red dot on a white canvas",
    
    # --- Fine-grained attributes ---
    "a wooden table with metal legs",
    "an old leather book with gold lettering",
    "a wet road reflecting streetlights at night",
    
    # --- Counting accuracy ---
    "exactly five red roses in a vase",
    "seven colored pencils in a row",
    "two birds on the left and three birds on the right",
    
    # --- Spatial relations (complex) ---
    "a cat inside a box with a ball on top of the box",
    "a person standing between two trees with mountains behind",
]

print(f"Total test prompts: {len(FAILURE_TEST_PROMPTS)}")

In [ ]:
def vlm_score_image(image, prompt, vlm_client, vlm_model):
    """Get a 0-10 score and description from VLM."""
    buffer = BytesIO()
    image.save(buffer, format='PNG')
    img_base64 = base64.b64encode(buffer.getvalue()).decode()
    
    eval_prompt = f"""Rate how well this image matches the description: "{prompt}"

Score on these criteria:
1. Object Presence (0-3): Are all objects present?
2. Attribute Accuracy (0-3): Are colors, sizes, textures correct?
3. Spatial Relations (0-2): Are positions/arrangements correct?
4. Overall Quality (0-2): Is the image realistic and coherent?

Return ONLY JSON: {{"total_score": X, "object_score": X, "attribute_score": X, "spatial_score": X, "quality_score": X, "notes": "brief explanation"}}"""
    
    try:
        response = vlm_client.chat.completions.create(
            model=vlm_model,
            messages=[{
                "role": "user",
                "content": [
                    {"type": "text", "text": eval_prompt},
                    {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{img_base64}"}},
                ],
            }],
            max_tokens=300,
        )
        result_text = response.choices[0].message.content
        json_match = re.search(r'\{[^}]+\}', result_text, re.DOTALL)
        if json_match:
            data = json.loads(json_match.group())
            total = float(data.get("total_score", 5))
            return total / 10.0, data  # normalize to [0,1]
        return 0.5, {"raw": result_text}
    except Exception as e:
        return 0.5, {"error": str(e)}

print("✓ VLM scorer defined")

In [ ]:
gen_config = GenerationConfig(seed=SEED, guidance_scale=5.0)

failure_results = []

for prompt in tqdm(FAILURE_TEST_PROMPTS, desc="Generating & Scoring"):
    # Generate
    baseline_imgs = baseline_gen.generate(prompt=prompt, config=gen_config)
    trained_imgs = trained_gen.generate(prompt=prompt, config=gen_config)
    
    # CLIP scores
    b_clip = clip_reward.compute_reward(baseline_imgs, [prompt]).rewards.item()
    t_clip = clip_reward.compute_reward(trained_imgs, [prompt]).rewards.item()
    
    # VLM scores
    b_vlm, b_vlm_details = vlm_score_image(baseline_imgs[0], prompt, vlm_client, VLM_MODEL)
    time.sleep(0.3)
    t_vlm, t_vlm_details = vlm_score_image(trained_imgs[0], prompt, vlm_client, VLM_MODEL)
    time.sleep(0.3)
    
    failure_results.append({
        "prompt": prompt,
        "baseline_img": baseline_imgs[0],
        "trained_img": trained_imgs[0],
        "b_clip": b_clip,
        "t_clip": t_clip,
        "b_vlm": b_vlm,
        "t_vlm": t_vlm,
        "b_vlm_details": b_vlm_details,
        "t_vlm_details": t_vlm_details,
    })
    
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"\n✓ Generated and scored {len(failure_results)} prompt pairs")

In [ ]:
# Identify failure modes: cases where trained is WORSE

degraded_cases = []  # Training made it worse
improved_cases = []  # Training helped
neutral_cases = []   # No significant change

THRESHOLD = 0.02  # Minimum score change to count as "different"

for r in failure_results:
    clip_delta = r['t_clip'] - r['b_clip']
    vlm_delta = r['t_vlm'] - r['b_vlm']
    avg_delta = (clip_delta + vlm_delta) / 2
    
    r['clip_delta'] = clip_delta
    r['vlm_delta'] = vlm_delta
    r['avg_delta'] = avg_delta
    
    if avg_delta < -THRESHOLD:
        degraded_cases.append(r)
    elif avg_delta > THRESHOLD:
        improved_cases.append(r)
    else:
        neutral_cases.append(r)

print(f"Improved:  {len(improved_cases)} ({len(improved_cases)/len(failure_results)*100:.0f}%)")
print(f"Degraded:  {len(degraded_cases)} ({len(degraded_cases)/len(failure_results)*100:.0f}%)")
print(f"Neutral:   {len(neutral_cases)} ({len(neutral_cases)/len(failure_results)*100:.0f}%)")

In [ ]:
# Show degraded cases (failure modes)
if degraded_cases:
    n = len(degraded_cases)
    fig, axes = plt.subplots(n, 2, figsize=(10, 4.5 * n))
    if n == 1:
        axes = axes.reshape(1, -1)
    
    fig.suptitle("FAILURE MODES — Training Made These Worse",
                 fontsize=16, fontweight='bold', color='red', y=1.02)
    
    # Sort by severity (worst degradation first)
    degraded_sorted = sorted(degraded_cases, key=lambda x: x['avg_delta'])
    
    for i, r in enumerate(degraded_sorted):
        import textwrap
        prompt_text = textwrap.fill(r['prompt'], width=35)
        
        axes[i, 0].imshow(r['baseline_img'])
        axes[i, 0].set_title(f"Baseline\nCLIP: {r['b_clip']:.3f} | VLM: {r['b_vlm']:.2f}",
                            fontsize=9, color='green')
        axes[i, 0].set_ylabel(prompt_text, fontsize=8, rotation=0,
                             labelpad=110, ha='right', va='center')
        axes[i, 0].set_xticks([])
        axes[i, 0].set_yticks([])
        
        axes[i, 1].imshow(r['trained_img'])
        axes[i, 1].set_title(
            f"Trained (WORSE)\nCLIP: {r['t_clip']:.3f} ({r['clip_delta']:+.3f}) | VLM: {r['t_vlm']:.2f} ({r['vlm_delta']:+.2f})",
            fontsize=9, color='red')
        axes[i, 1].set_xticks([])
        axes[i, 1].set_yticks([])
        # Red border for degraded
        for spine in axes[i, 1].spines.values():
            spine.set_edgecolor('red')
            spine.set_linewidth(3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'failure_modes.png'), dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("No degraded cases found! Training improved or maintained all prompts.")

---
## Part 2: Reward Hacking Detection

**Reward hacking** = the model learns to exploit the reward signal without actually improving.

We detect this by finding cases where:
- CLIP score is high but VLM score is low (or vice versa)
- The trained model has high reward but the VLM-judged image quality is bad

Key indicators:
1. CLIP-VLM disagreement (high CLIP but low VLM)
2. Degenerate outputs (e.g., blurry, repetitive textures, artifacts)

In [ ]:
# Scatter plot: CLIP vs VLM for both baseline and trained

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- Left: Baseline ---
b_clips = [r['b_clip'] for r in failure_results]
b_vlms = [r['b_vlm'] for r in failure_results]

axes[0].scatter(b_clips, b_vlms, c='#FF6B6B', s=60, alpha=0.7, edgecolors='black', linewidths=0.5)
axes[0].set_xlabel('CLIP Score', fontsize=11)
axes[0].set_ylabel('VLM Score', fontsize=11)
axes[0].set_title('Baseline: CLIP vs VLM', fontsize=13, fontweight='bold')
axes[0].set_xlim(0, 0.5)
axes[0].set_ylim(0, 1.05)
axes[0].plot([0, 1], [0, 2], 'k--', alpha=0.3, label='perfect correlation')  # rough reference

# --- Right: Trained ---
t_clips = [r['t_clip'] for r in failure_results]
t_vlms = [r['t_vlm'] for r in failure_results]

axes[1].scatter(t_clips, t_vlms, c='#4ECDC4', s=60, alpha=0.7, edgecolors='black', linewidths=0.5)
axes[1].set_xlabel('CLIP Score', fontsize=11)
axes[1].set_ylabel('VLM Score', fontsize=11)
axes[1].set_title('Trained: CLIP vs VLM', fontsize=13, fontweight='bold')
axes[1].set_xlim(0, 0.5)
axes[1].set_ylim(0, 1.05)
axes[1].plot([0, 1], [0, 2], 'k--', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'clip_vs_vlm_scatter.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Detect potential reward hacking: high CLIP improvement but low/negative VLM change

reward_hacking_candidates = []

for r in failure_results:
    clip_delta = r['t_clip'] - r['b_clip']
    vlm_delta = r['t_vlm'] - r['b_vlm']
    
    # Case 1: CLIP improved but VLM got worse
    if clip_delta > 0.01 and vlm_delta < -0.05:
        r['hack_type'] = 'CLIP↑ VLM↓'
        reward_hacking_candidates.append(r)
    # Case 2: Both CLIP and VLM are high for trained, but VLM notes issues
    elif r['t_clip'] > 0.28 and r['t_vlm'] < 0.4:
        r['hack_type'] = 'High CLIP, Low VLM'
        reward_hacking_candidates.append(r)

print(f"Potential reward hacking cases: {len(reward_hacking_candidates)} / {len(failure_results)}")

if reward_hacking_candidates:
    for r in reward_hacking_candidates:
        print(f"\n  Prompt: {r['prompt'][:60]}")
        print(f"    Type: {r['hack_type']}")
        print(f"    CLIP: {r['b_clip']:.3f} → {r['t_clip']:.3f} (Δ={r['clip_delta']:+.3f})")
        print(f"    VLM:  {r['b_vlm']:.2f} → {r['t_vlm']:.2f} (Δ={r['vlm_delta']:+.2f})")
else:
    print("No obvious reward hacking detected.")

In [ ]:
# Visualize reward hacking candidates (if any)

if reward_hacking_candidates:
    n = min(len(reward_hacking_candidates), 6)  # Show max 6
    fig, axes = plt.subplots(n, 2, figsize=(10, 4.5 * n))
    if n == 1:
        axes = axes.reshape(1, -1)
    
    fig.suptitle("Potential Reward Hacking Cases",
                 fontsize=16, fontweight='bold', color='darkorange', y=1.02)
    
    for i, r in enumerate(reward_hacking_candidates[:n]):
        import textwrap
        prompt_text = textwrap.fill(r['prompt'], width=35)
        hack_label = r['hack_type']
        
        axes[i, 0].imshow(r['baseline_img'])
        axes[i, 0].set_title(f"Baseline\nCLIP: {r['b_clip']:.3f} | VLM: {r['b_vlm']:.2f}",
                            fontsize=9)
        axes[i, 0].set_ylabel(f"[{hack_label}]\n{prompt_text}", fontsize=7,
                             rotation=0, labelpad=110, ha='right', va='center')
        axes[i, 0].set_xticks([])
        axes[i, 0].set_yticks([])
        
        axes[i, 1].imshow(r['trained_img'])
        axes[i, 1].set_title(
            f"Trained\nCLIP: {r['t_clip']:.3f} ({r['clip_delta']:+.3f}) | VLM: {r['t_vlm']:.2f} ({r['vlm_delta']:+.2f})",
            fontsize=9, color='darkorange')
        axes[i, 1].set_xticks([])
        axes[i, 1].set_yticks([])
        for spine in axes[i, 1].spines.values():
            spine.set_edgecolor('orange')
            spine.set_linewidth(3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'reward_hacking_cases.png'), dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("No reward hacking cases to visualize.")

In [ ]:
# Delta analysis chart: visualize improvement/degradation per prompt

fig, ax = plt.subplots(figsize=(14, 7))

prompts_short = [r['prompt'][:30] + '...' if len(r['prompt']) > 30 else r['prompt']
                 for r in failure_results]
clip_deltas = [r['clip_delta'] for r in failure_results]
vlm_deltas = [r['vlm_delta'] for r in failure_results]

x = np.arange(len(failure_results))
width = 0.35

# Color bars by sign
clip_colors = ['#4ECDC4' if d > 0 else '#FF6B6B' for d in clip_deltas]
vlm_colors = ['#45B7D1' if d > 0 else '#FFA07A' for d in vlm_deltas]

bars1 = ax.bar(x - width/2, clip_deltas, width, color=clip_colors, alpha=0.85, label='CLIP Δ')
bars2 = ax.bar(x + width/2, vlm_deltas, width, color=vlm_colors, alpha=0.85, label='VLM Δ')

ax.axhline(y=0, color='black', linewidth=0.5)
ax.set_ylabel('Score Change (Trained - Baseline)', fontsize=11)
ax.set_title('Per-Prompt Score Change After Training', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(prompts_short, rotation=60, ha='right', fontsize=7)
ax.legend(fontsize=10)

# Add reference regions
ax.axhspan(0, ax.get_ylim()[1], alpha=0.05, color='green')
ax.axhspan(ax.get_ylim()[0], 0, alpha=0.05, color='red')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'per_prompt_delta.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## Part 3: Epoch-by-Epoch Progression

Track how generation changes across training checkpoints (if available).

**Skip this section if epoch checkpoints are not available** (set `EPOCH_CHECKPOINTS = {}` in Step 0).

In [ ]:
# Prompts for epoch progression analysis
PROGRESSION_PROMPTS = [
    "a red apple and a blue cup",
    "three cats on a sofa",
    "a bird above the tree",
    "a large elephant and a small mouse",
    "a purple flower in a pink vase",
]

if not EPOCH_CHECKPOINTS:
    print("Epoch checkpoints not configured. Skipping progression analysis.")
    print("Set EPOCH_CHECKPOINTS in Step 0 to enable this section.")
    progression_data = None
else:
    # Check which checkpoints actually exist
    available_checkpoints = {}
    for name, path in EPOCH_CHECKPOINTS.items():
        if os.path.exists(path):
            available_checkpoints[name] = path
            print(f"  ✓ {name}: {path}")
        else:
            print(f"  ✗ {name}: {path} (NOT FOUND)")
    
    if not available_checkpoints:
        print("\nNo checkpoint directories found. Skipping progression.")
        progression_data = None
    else:
        print(f"\nUsing {len(available_checkpoints)} checkpoints for progression analysis.")
        progression_data = available_checkpoints

In [ ]:
if progression_data:
    # Generate images at each checkpoint
    # We reuse the baseline generator and swap LoRA weights
    
    epoch_gen = JanusProGenerator(
        model_name_or_path=BASELINE_MODEL,
        device="cuda",
        load_in_4bit=USE_4BIT,
    )
    epoch_gen.load_model()
    
    gen_config_prog = GenerationConfig(seed=SEED, guidance_scale=5.0)
    
    # Results: {prompt -> {checkpoint_name -> {img, clip, vlm}}}
    epoch_results = {p: {} for p in PROGRESSION_PROMPTS}
    
    # First, generate baseline images (no LoRA)
    print("Generating baseline images...")
    for prompt in PROGRESSION_PROMPTS:
        imgs = baseline_gen.generate(prompt=prompt, config=gen_config_prog)
        clip_s = clip_reward.compute_reward(imgs, [prompt]).rewards.item()
        epoch_results[prompt]["Baseline"] = {"img": imgs[0], "clip": clip_s}
    
    # Then generate for each checkpoint
    for ckpt_name, ckpt_path in progression_data.items():
        print(f"\nLoading checkpoint: {ckpt_name} ({ckpt_path})")
        try:
            # Reload model fresh to avoid stacking LoRA
            epoch_gen_ckpt = JanusProGenerator(
                model_name_or_path=BASELINE_MODEL,
                device="cuda",
                load_in_4bit=USE_4BIT,
            )
            epoch_gen_ckpt.load_model()
            epoch_gen_ckpt.enable_lora(lora_path=ckpt_path)
            epoch_gen_ckpt.eval()
            
            for prompt in PROGRESSION_PROMPTS:
                imgs = epoch_gen_ckpt.generate(prompt=prompt, config=gen_config_prog)
                clip_s = clip_reward.compute_reward(imgs, [prompt]).rewards.item()
                epoch_results[prompt][ckpt_name] = {"img": imgs[0], "clip": clip_s}
            
            # Free memory
            del epoch_gen_ckpt
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        except Exception as e:
            print(f"  Error loading {ckpt_name}: {e}")
    
    print("\n✓ Epoch progression images generated")
else:
    epoch_results = None
    print("Skipping epoch progression (no checkpoints).")

In [ ]:
if epoch_results:
    # Plot progression grid: rows = prompts, cols = checkpoints
    import textwrap
    
    all_ckpt_names = ["Baseline"] + list(progression_data.keys())
    # Filter to only checkpoints that have data
    all_ckpt_names = [c for c in all_ckpt_names 
                      if c in epoch_results[PROGRESSION_PROMPTS[0]]]
    n_prompts = len(PROGRESSION_PROMPTS)
    n_ckpts = len(all_ckpt_names)
    
    fig, axes = plt.subplots(n_prompts, n_ckpts, figsize=(3.5 * n_ckpts, 3.5 * n_prompts))
    
    fig.suptitle("Epoch-by-Epoch Generation Progression",
                 fontsize=16, fontweight='bold', y=1.02)
    
    for j, ckpt_name in enumerate(all_ckpt_names):
        axes[0, j].set_title(ckpt_name, fontsize=11, fontweight='bold')
    
    for i, prompt in enumerate(PROGRESSION_PROMPTS):
        prompt_text = textwrap.fill(prompt, width=25)
        axes[i, 0].set_ylabel(prompt_text, fontsize=8, rotation=0,
                             labelpad=90, ha='right', va='center')
        
        for j, ckpt_name in enumerate(all_ckpt_names):
            data = epoch_results[prompt].get(ckpt_name)
            if data:
                axes[i, j].imshow(data['img'])
                axes[i, j].text(0.02, 0.02, f"CLIP: {data['clip']:.3f}",
                               transform=axes[i, j].transAxes, fontsize=7,
                               color='white', backgroundcolor='black', va='bottom')
            else:
                axes[i, j].text(0.5, 0.5, 'N/A', transform=axes[i, j].transAxes,
                               ha='center', fontsize=12)
            axes[i, j].set_xticks([])
            axes[i, j].set_yticks([])
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'epoch_progression.png'), dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("Skipping epoch progression plot.")

In [ ]:
if epoch_results:
    # CLIP score progression over epochs
    all_ckpt_names = ["Baseline"] + list(progression_data.keys())
    all_ckpt_names = [c for c in all_ckpt_names 
                      if c in epoch_results[PROGRESSION_PROMPTS[0]]]
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    for prompt in PROGRESSION_PROMPTS:
        scores = []
        for ckpt in all_ckpt_names:
            data = epoch_results[prompt].get(ckpt)
            scores.append(data['clip'] if data else None)
        
        label = prompt[:40] + '...' if len(prompt) > 40 else prompt
        ax.plot(range(len(scores)), scores, 'o-', label=label, markersize=6)
    
    ax.set_xlabel('Checkpoint', fontsize=11)
    ax.set_ylabel('CLIP Score', fontsize=11)
    ax.set_title('CLIP Score Progression Across Epochs', fontsize=14, fontweight='bold')
    ax.set_xticks(range(len(all_ckpt_names)))
    ax.set_xticklabels(all_ckpt_names)
    ax.legend(fontsize=8, loc='best')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'clip_progression.png'), dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("Skipping progression plot.")

---
## Part 4: Reward Signal Case Study

Visualize the relationship between reward values and actual image quality.

Show concrete examples of reward-quality correlation and mismatches.

In [ ]:
# Build a comprehensive table of all results

rows = []
for r in failure_results:
    rows.append({
        "Prompt": r['prompt'][:45] + ('...' if len(r['prompt']) > 45 else ''),
        "B_CLIP": round(r['b_clip'], 4),
        "T_CLIP": round(r['t_clip'], 4),
        "CLIP_Δ": round(r['clip_delta'], 4),
        "B_VLM": round(r['b_vlm'], 3),
        "T_VLM": round(r['t_vlm'], 3),
        "VLM_Δ": round(r['vlm_delta'], 3),
        "Agree": "✓" if (r['clip_delta'] > 0 and r['vlm_delta'] > 0) or
                        (r['clip_delta'] < 0 and r['vlm_delta'] < 0) or
                        (abs(r['clip_delta']) < 0.01 and abs(r['vlm_delta']) < 0.05)
                 else "✗",
    })

df_all = pd.DataFrame(rows)
print("\n" + "=" * 110)
print("COMPREHENSIVE REWARD & QUALITY ANALYSIS")
print("=" * 110)
with pd.option_context('display.max_colwidth', 45, 'display.width', 200):
    print(df_all.to_string(index=False))

# Correlation between CLIP and VLM
from scipy import stats
try:
    r_base, p_base = stats.pearsonr(b_clips, b_vlms)
    r_train, p_train = stats.pearsonr(t_clips, t_vlms)
    print(f"\nCorrelation (CLIP vs VLM):")
    print(f"  Baseline: r={r_base:.3f} (p={p_base:.4f})")
    print(f"  Trained:  r={r_train:.3f} (p={p_train:.4f})")
except:
    print("Need scipy for correlation. Install: pip install scipy")

# Agreement rate
agree_count = sum(1 for r in rows if r['Agree'] == '✓')
print(f"\nCLIP-VLM Agreement: {agree_count}/{len(rows)} ({agree_count/len(rows)*100:.0f}%)")

In [ ]:
# Best and worst examples side by side

# Sort by average improvement
sorted_results = sorted(failure_results, key=lambda x: x['avg_delta'], reverse=True)

n_show = 3  # Show top-3 best and worst

best_cases = sorted_results[:n_show]
worst_cases = sorted_results[-n_show:]

fig, axes = plt.subplots(n_show * 2, 2, figsize=(10, 4 * n_show * 2))

fig.suptitle("Best Improvements vs Worst Degradations",
             fontsize=16, fontweight='bold', y=1.01)

import textwrap

for i, r in enumerate(best_cases):
    prompt_text = textwrap.fill(r['prompt'], width=35)
    
    axes[i, 0].imshow(r['baseline_img'])
    axes[i, 0].set_title(f"Baseline | CLIP: {r['b_clip']:.3f} | VLM: {r['b_vlm']:.2f}", fontsize=9)
    axes[i, 0].set_ylabel(f"✓ BEST #{i+1}\n{prompt_text}", fontsize=7,
                         rotation=0, labelpad=110, ha='right', va='center',
                         color='green')
    axes[i, 0].set_xticks([])
    axes[i, 0].set_yticks([])
    
    axes[i, 1].imshow(r['trained_img'])
    axes[i, 1].set_title(
        f"Trained | CLIP: {r['t_clip']:.3f} ({r['clip_delta']:+.3f}) | VLM: {r['t_vlm']:.2f} ({r['vlm_delta']:+.2f})",
        fontsize=9, color='green')
    axes[i, 1].set_xticks([])
    axes[i, 1].set_yticks([])
    for spine in axes[i, 1].spines.values():
        spine.set_edgecolor('green')
        spine.set_linewidth(2)

for i, r in enumerate(worst_cases):
    row_idx = n_show + i
    prompt_text = textwrap.fill(r['prompt'], width=35)
    
    axes[row_idx, 0].imshow(r['baseline_img'])
    axes[row_idx, 0].set_title(f"Baseline | CLIP: {r['b_clip']:.3f} | VLM: {r['b_vlm']:.2f}", fontsize=9)
    axes[row_idx, 0].set_ylabel(f"✗ WORST #{i+1}\n{prompt_text}", fontsize=7,
                               rotation=0, labelpad=110, ha='right', va='center',
                               color='red')
    axes[row_idx, 0].set_xticks([])
    axes[row_idx, 0].set_yticks([])
    
    axes[row_idx, 1].imshow(r['trained_img'])
    axes[row_idx, 1].set_title(
        f"Trained | CLIP: {r['t_clip']:.3f} ({r['clip_delta']:+.3f}) | VLM: {r['t_vlm']:.2f} ({r['vlm_delta']:+.2f})",
        fontsize=9, color='red')
    axes[row_idx, 1].set_xticks([])
    axes[row_idx, 1].set_yticks([])
    for spine in axes[row_idx, 1].spines.values():
        spine.set_edgecolor('red')
        spine.set_linewidth(2)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'best_vs_worst.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## Part 5: Summary Statistics & Save

In [ ]:
# Print final summary

print("=" * 70)
print("FAILURE MODE ANALYSIS — SUMMARY")
print("=" * 70)

print(f"\nTotal prompts tested: {len(failure_results)}")
print(f"  Improved by training: {len(improved_cases)} ({len(improved_cases)/len(failure_results)*100:.0f}%)")
print(f"  Degraded by training: {len(degraded_cases)} ({len(degraded_cases)/len(failure_results)*100:.0f}%)")
print(f"  Neutral/unchanged:    {len(neutral_cases)} ({len(neutral_cases)/len(failure_results)*100:.0f}%)")

avg_clip_delta = np.mean([r['clip_delta'] for r in failure_results])
avg_vlm_delta = np.mean([r['vlm_delta'] for r in failure_results])
print(f"\nAverage score change:")
print(f"  CLIP: {avg_clip_delta:+.4f}")
print(f"  VLM:  {avg_vlm_delta:+.3f}")

print(f"\nReward hacking candidates: {len(reward_hacking_candidates)}")

if degraded_cases:
    print(f"\nDegraded prompts (failure modes):")
    for r in sorted(degraded_cases, key=lambda x: x['avg_delta']):
        print(f"  • \"{r['prompt'][:50]}\" (avg Δ = {r['avg_delta']:+.3f})")

print(f"\nEpoch progression: {'Available' if epoch_results else 'Not available'}")

In [ ]:
# Save all results

save_data = {
    "summary": {
        "total_prompts": len(failure_results),
        "improved": len(improved_cases),
        "degraded": len(degraded_cases),
        "neutral": len(neutral_cases),
        "avg_clip_delta": float(avg_clip_delta),
        "avg_vlm_delta": float(avg_vlm_delta),
        "reward_hacking_count": len(reward_hacking_candidates),
    },
    "per_prompt": [
        {
            "prompt": r['prompt'],
            "b_clip": r['b_clip'],
            "t_clip": r['t_clip'],
            "clip_delta": r['clip_delta'],
            "b_vlm": r['b_vlm'],
            "t_vlm": r['t_vlm'],
            "vlm_delta": r['vlm_delta'],
            "b_vlm_notes": r['b_vlm_details'].get('notes', ''),
            "t_vlm_notes": r['t_vlm_details'].get('notes', ''),
        }
        for r in failure_results
    ],
    "degraded_prompts": [r['prompt'] for r in degraded_cases],
    "reward_hacking_prompts": [
        {"prompt": r['prompt'], "type": r.get('hack_type', '')}
        for r in reward_hacking_candidates
    ],
    "config": {
        "baseline_model": BASELINE_MODEL,
        "lora_path": LORA_WEIGHTS_PATH,
        "vlm_model": VLM_MODEL,
        "seed": SEED,
    }
}

with open(os.path.join(OUTPUT_DIR, 'failure_mode_results.json'), 'w') as f:
    json.dump(save_data, f, indent=2, default=str)

df_all.to_csv(os.path.join(OUTPUT_DIR, 'reward_quality_table.csv'), index=False)

print(f"\n✓ Results saved to {OUTPUT_DIR}/")
for f_name in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f_name))
    print(f"  {f_name} ({size/1024:.1f} KB)")

In [ ]:
# Download as zip
!cd {OUTPUT_DIR} && zip -r ../failure_mode_results.zip .
try:
    from google.colab import files
    files.download('failure_mode_results.zip')
except ImportError:
    print("Not in Colab. Files saved locally in:", OUTPUT_DIR)